In [1]:
import re
import copy
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)

PyTorch version: 2.11.0+cu128
Device: cuda


In [2]:
dataset = load_dataset(
    "PolyAI/banking77",
    trust_remote_code=True
)

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_feature = dataset["train"].features["label"]

X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["label"]
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test giữ nguyên:", len(test_df))
print("Số intent:", y_train.nunique())

Train: 8502
Validation: 1501
Test giữ nguyên: 3080
Số intent: 77


In [3]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_IDX = 0
UNK_IDX = 1

MAX_LENGTH = 40
MIN_FREQ = 2

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def build_vocab(texts, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(tokenize(text))

    vocab = {
        PAD_TOKEN: PAD_IDX,
        UNK_TOKEN: UNK_IDX
    }

    for token, frequency in counter.items():
        if frequency >= min_freq:
            vocab[token] = len(vocab)

    return vocab

vocab = build_vocab(X_train, min_freq=MIN_FREQ)

def encode_text(text, vocab, max_length=40):
    token_ids = [
        vocab.get(token, UNK_IDX)
        for token in tokenize(text)
    ]

    token_ids = token_ids[:max_length]

    padding_length = max_length - len(token_ids)
    token_ids += [PAD_IDX] * padding_length

    return token_ids

print("Vocabulary size:", len(vocab))
print("Ví dụ:")
print(X_train.iloc[0])
print(encode_text(X_train.iloc[0], vocab)[:15])

Vocabulary size: 1371
Ví dụ:
I don't have my passcode to access the app.
[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0, 0, 0, 0, 0]


In [4]:
class BankingTicketDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.texts = texts.tolist()
        self.labels = [int(label) for label in labels.tolist()]
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        token_ids = encode_text(
            self.texts[index],
            self.vocab,
            self.max_length
        )

        label = self.labels[index]

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

train_dataset = BankingTicketDataset(
    X_train, y_train, vocab, MAX_LENGTH
)

val_dataset = BankingTicketDataset(
    X_val, y_val, vocab, MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=0
)

tokens, labels = next(iter(train_loader))

print("Token batch shape:", tokens.shape)
print("Label batch shape:", labels.shape)

Token batch shape: torch.Size([64, 40])
Label batch shape: torch.Size([64])


In [5]:
example = "My card is blocked!"

print("Text:", example)
print("Tokens:", tokenize(example))

encoded = encode_text(example, vocab, MAX_LENGTH)

print("Token IDs:", encoded)
print("Length:", len(encoded))

Text: My card is blocked!
Tokens: ['my', 'card', 'is', 'blocked']
Token IDs: [6, 40, 37, 402, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Length: 40


In [7]:
class TextCNN(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        num_classes,
        filter_sizes=(3, 4, 5),
        num_filters=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=filter_size
            )
            for filter_size in filter_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            num_filters * len(filter_sizes),
            num_classes
        )

    def forward(self, token_ids):
        # [batch_size, max_length]
        embedded = self.embedding(token_ids)

        # [batch_size, max_length, embedding_dim]
        embedded = embedded.transpose(1, 2)

        # [batch_size, embedding_dim, max_length]
        pooled_features = []

        for conv in self.convs:
            feature_map = torch.relu(conv(embedded))
            pooled = torch.max(feature_map, dim=2).values
            pooled_features.append(pooled)

        combined = torch.cat(pooled_features, dim=1)
        combined = self.dropout(combined)

        logits = self.classifier(combined)

        return logits

In [8]:
NUM_CLASSES = y_train.nunique()

model = TextCNN(
    vocab_size=len(vocab),
    embedding_dim=128,
    num_classes=NUM_CLASSES,
    filter_sizes=(3, 4, 5),
    num_filters=128,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(model)

TextCNN(
  (embedding): Embedding(1371, 128, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
    (1): Conv1d(128, 128, kernel_size=(4,), stride=(1,))
    (2): Conv1d(128, 128, kernel_size=(5,), stride=(1,))
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (classifier): Linear(in_features=384, out_features=77, bias=True)
)


In [9]:
def evaluate(model, data_loader):
    model.eval()

    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for token_ids, labels in data_loader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)

            logits = model(token_ids)
            predictions = torch.argmax(logits, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_predictions)
    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return accuracy, macro_f1, all_labels, all_predictions


def train_model(model, train_loader, val_loader, criterion, optimizer,
                epochs=12, patience=3):
    best_macro_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        for token_ids, labels in train_loader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(token_ids)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * labels.size(0)

        train_loss = total_loss / len(train_loader.dataset)

        val_accuracy, val_macro_f1, _, _ = evaluate(
            model, val_loader
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": val_accuracy,
            "val_macro_f1": val_macro_f1
        })

        print(
            f"Epoch {epoch:02d} | "
            f"loss: {train_loss:.4f} | "
            f"val accuracy: {val_accuracy:.4f} | "
            f"val Macro-F1: {val_macro_f1:.4f}"
        )

        if val_macro_f1 > best_macro_f1:
            best_macro_f1 = val_macro_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

            if epochs_without_improvement >= patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)

    return pd.DataFrame(history), best_macro_f1


history_df, best_val_macro_f1 = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=12,
    patience=3
)

Epoch 01 | loss: 2.9594 | val accuracy: 0.6342 | val Macro-F1: 0.6100
Epoch 02 | loss: 1.2077 | val accuracy: 0.7648 | val Macro-F1: 0.7575
Epoch 03 | loss: 0.7523 | val accuracy: 0.7895 | val Macro-F1: 0.7808
Epoch 04 | loss: 0.5276 | val accuracy: 0.8121 | val Macro-F1: 0.8079
Epoch 05 | loss: 0.4086 | val accuracy: 0.8301 | val Macro-F1: 0.8251
Epoch 06 | loss: 0.3189 | val accuracy: 0.8341 | val Macro-F1: 0.8250
Epoch 07 | loss: 0.2647 | val accuracy: 0.8468 | val Macro-F1: 0.8379
Epoch 08 | loss: 0.2134 | val accuracy: 0.8454 | val Macro-F1: 0.8380
Epoch 09 | loss: 0.1912 | val accuracy: 0.8501 | val Macro-F1: 0.8439
Epoch 10 | loss: 0.1610 | val accuracy: 0.8441 | val Macro-F1: 0.8381
Epoch 11 | loss: 0.1468 | val accuracy: 0.8481 | val Macro-F1: 0.8396
Epoch 12 | loss: 0.1284 | val accuracy: 0.8521 | val Macro-F1: 0.8439
Early stopping.


In [10]:
textcnn_accuracy, textcnn_macro_f1, y_val_true, textcnn_val_pred = evaluate(
    model,
    val_loader
)

print(f"TextCNN validation accuracy: {textcnn_accuracy:.4f}")
print(f"TextCNN validation Macro-F1: {textcnn_macro_f1:.4f}")

comparison_df = pd.DataFrame({
    "model": ["Naive Bayes", "Linear SVM", "TextCNN"],
    "accuracy": [
        0.797468,
        0.876749,
        textcnn_accuracy
    ],
    "macro_f1": [
        0.763062,
        0.869111,
        textcnn_macro_f1
    ]
})

display(comparison_df.sort_values("macro_f1", ascending=False))

TextCNN validation accuracy: 0.8501
TextCNN validation Macro-F1: 0.8439


,model,accuracy,macro_f1
1,Linear SVM,0.876749,0.869111
2,TextCNN,0.850100,0.843888
0,Naive Bayes,0.797468,0.763062
